In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from datasets import load_dataset

model = SentenceTransformer('all-MiniLM-L6-v2')
dataset = load_dataset("sentence-transformers/stsb")

In [ ]:
embeddings = model.encode(["some sentence", "another sentence", "I like dogs", "I like cats"])
cosine_similarity([embeddings[0]], [embeddings[3]]).item()

In [ ]:
train = dataset['train']
dataset['train'][:5]  # returns a dict of lists for the first 5 rows

In [ ]:
# Inference
emb1 = model.encode(train['sentence1'])
emb2 = model.encode(train['sentence2'])

In [ ]:
cosine_similarity([emb1[4]], [emb2[4]]).item()

## STSb evaluation

Embed every pair in the test split, take the cosine similarity of each pair, and report the Spearman correlation against the human scores. `all-MiniLM-L6-v2` should land around ~0.82 — a sanity check that the pipeline matches the standard benchmark.

In [ ]:
from scipy.stats import spearmanr

test = dataset['test']

# unit-normalize so a row-wise dot product is the cosine similarity
emb1 = model.encode(test['sentence1'], normalize_embeddings=True)
emb2 = model.encode(test['sentence2'], normalize_embeddings=True)
cos_sims = (emb1 * emb2).sum(axis=1)

spearman = spearmanr(cos_sims, test['score']).statistic
print(f"Spearman correlation: {spearman:.4f}")